In [1]:
# Permite ajustar la anchura de la parte útil de la libreta (reduce los márgenes)
from IPython.core.display import display, HTML
display(HTML("<style>.container{ width:90% }</style>"))

/tmp/ipykernel_1049099/1126351303.py:2: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


# Setup installs & imports

In [2]:
# ## install required packages
# !pip install swig
# !pip install wrds
# !pip install pyportfolioopt
# ## install finrl library
# !pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

In [3]:
import os
import pandas as pd

from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent
from stable_baselines3.common.logger import configure
from finrl import config_tickers
from finrl.main import check_and_make_directories
from finrl.config import INDICATORS, TRAINED_MODEL_DIR, RESULTS_DIR

check_and_make_directories([TRAINED_MODEL_DIR])

2025-05-19 11:17:58.334249: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-19 11:17:58.349661: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-19 11:17:58.368801: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-19 11:17:58.374611: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-19 11:17:58.388901: I tensorflow/core/platform/cpu_feature_guar

In [4]:
import pickle

with open("train.pkl", "rb") as f:
    train = pickle.load(f)

In [5]:
train.head()

,date,tic,close,high,low,open,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
0,2009-01-02,aapl,2.727417,2.736134,2.559415,2.581054,746015200.0,4.0,0.0,2.947757,2.622185,100.0,66.666667,100.0,2.727417,2.727417,39.189999,0.0
0,2009-01-02,amzn,2.718000,2.726500,2.553500,2.567500,145928000.0,4.0,0.0,2.947757,2.622185,100.0,66.666667,100.0,2.718000,2.718000,39.189999,0.0
0,2009-01-02,cat,30.344679,30.389960,28.921564,29.050939,7117200.0,4.0,0.0,2.947757,2.622185,100.0,66.666667,100.0,30.344679,30.344679,39.189999,0.0
0,2009-01-02,hd,16.148619,16.188774,15.365615,15.439231,14902500.0,4.0,0.0,2.947757,2.622185,100.0,66.666667,100.0,16.148619,16.148619,39.189999,0.0
0,2009-01-02,ibm,47.715294,47.835443,45.814766,45.814766,7905877.0,4.0,0.0,2.947757,2.622185,100.0,66.666667,100.0,47.715294,47.715294,39.189999,0.0


# Load the Data from Part 1

In [6]:
# # Quito el índice a la izquierda de la fecha

# #train = pd.read_csv('')
# train = train.set_index(train.columns[0])
# train.index.names = ['']

# train

## Setup Environment

### 📊 Representación del Estado en un Entorno de Trading (Reinforcement Learning)

En Aprendizaje por Refuerzo (RL), el **estado** representa toda la información que el agente necesita para tomar decisiones óptimas.

---

### 🧩 ¿Qué incluye el estado en este entorno de trading?

Cada estado está compuesto por varios elementos clave que reflejan tanto la situación financiera del agente como el estado del mercado:

| Parte                                  | Elemento                   | Descripción                                                               |
|----------------------------------------|-----------------------------|---------------------------------------------------------------------------|
| `1`                                    | `cash`                      | Cantidad de efectivo disponible (1 valor escalar).                        |
| `stock_dimension`                      | `acciones en posesión`      | Número de acciones que el agente posee por cada ticker.                   |
| `stock_dimension`                      | `precios de acciones`       | Precio actual de cada acción.                                             |
| `stock_dimension × len(INDICATORS)`    | `indicadores técnicos`      | Valores de indicadores (como MACD, RSI, etc.) para cada acción.          |

---

### 🧮 Fórmula de cálculo del tamaño del estado (`state_space`)


$$ state\_space = 1 + 2 * stock\_dimension + len(INDICATORS) * stock\_dimension$$


---

### 🔍 Desglose:

- `1`: efectivo  
- `+ stock_dimension`: cantidad de acciones en portafolio  
- `+ stock_dimension`: precios actuales de cada acción  
- `+ len(INDICATORS) * stock_dimension`: indicadores técnicos por acción  

---

### 🧠 ¿Por qué se incluye cada parte?

Porque en trading, el agente necesita saber:

#### 💰 Qué tiene:
- Efectivo disponible  
- Acciones en posesión  

#### 📈 Cuál es el estado del mercado:
- Precios actuales de las acciones  

#### 📊 Qué señales existen para tomar decisiones:
- Indicadores técnicos como MACD, RSI, CCI, etc.

Con esa información, el agente puede decidir si **compra**, **vende** o **mantiene** una acción.


In [7]:
stock_dimension = len(train.tic.unique()) # tickers únicos sin repetir

state_space = 1 + 2*stock_dimension + len(INDICATORS)*stock_dimension

print(f"Number of Indicators: {len(INDICATORS)}")

print(f"Stock Dimension: {stock_dimension}")

print(f"state_space = 1 + 2*{stock_dimension} + ({len(INDICATORS)})*{stock_dimension}")

print(f"State Space: {state_space}")

Number of Indicators: 8
Stock Dimension: 8
state_space = 1 + 2*8 + (8)*8
State Space: 81


### 📈 Interpretación de los valores: `Stock Dimension` y `State Space`

| Valor                | Descripción                                                                 |
|----------------------|-----------------------------------------------------------------------------|
| **Stock Dimension: 8** | Tienes **8 acciones distintas**, es decir, 8 símbolos únicos en la columna `tic`. |
| **State Space: 81**    | En cada paso del entorno, el agente recibe un **vector de estado de longitud 81**, que incluye:  
- 💰 Efectivo disponible  
- 📦 Cantidad de acciones en posesión (una por cada ticker)  
- 📈 Precios actuales de mercado (uno por cada ticker)  
- 🧠 Indicadores técnicos (como RSI, MACD, etc.) para cada acción |

In [8]:
# Definimos las comisiones de compra y venta como listas de 0.1% por cada acción del portafolio
buy_cost_list = sell_cost_list = [0.001] * stock_dimension

# Inicializamos que el agente no posee ninguna acción de ningún ticker al comenzar
num_stock_shares = [0] * stock_dimension

# Creamos un diccionario con todos los parámetros del entorno
env_kwargs = {
    # Número máximo de acciones que el agente puede comprar o vender en un solo paso
    "hmax": 100,

    # Monto de dinero inicial con el que empieza el agente ($1,000,000)
    "initial_amount": 1000000,

    # Cantidad de acciones que posee el agente al inicio (todas en cero)
    "num_stock_shares": num_stock_shares,

    # Porcentaje de comisión al comprar acciones
    "buy_cost_pct": buy_cost_list,

    # Porcentaje de comisión al vender acciones
    "sell_cost_pct": sell_cost_list,

    # Dimensión del vector de estado (efectivo, acciones, precios e indicadores)
    "state_space": state_space,

    # Número de acciones distintas en el portafolio
    "stock_dim": stock_dimension,

    # Lista de indicadores técnicos usados como parte del estado
    "tech_indicator_list": INDICATORS,

    # Dimensión del espacio de acciones (una acción por ticker)
    "action_space": stock_dimension,

    # Factor de escalado de la recompensa para mejorar la estabilidad del entrenamiento
    "reward_scaling": 1e-4
}

# Creamos una instancia del entorno StockTradingEnv usando los datos procesados en 'train'
# Este entorno simula un mercado financiero para que un agente de RL pueda aprender a invertir
e_train_gym = StockTradingEnv(df=train, **env_kwargs)

In [9]:
e_train_gym

### Environment for Training

In [10]:
env_train, _ = e_train_gym.get_sb_env()
print(type(env_train))

<class 'stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv'>


# Train Agent

In [11]:
agent = DRLAgent(env = env_train)

# Set the corresponding values to 'True' for the algorithms that you want to use
if_using_a2c = False
if_using_ddpg = False
if_using_ppo = True
if_using_td3 = False
if_using_sac = False

## A2C Model

The code above and below provide a healthy framework to add different agent types for training within the same notebook. For this I'll only be training an a2c model, but may add ddpg, ppo, and td3 in later iterations.

In [12]:
model_a2c = agent.get_model("a2c")
model_ppo = agent.get_model('ppo')

if if_using_a2c:
  # set up logger
  tmp_path = RESULTS_DIR + '/a2c'
  new_logger_a2c = configure(tmp_path, ["stdout", "csv", "tensorboard"])
  # Set new logger
  model_a2c.set_logger(new_logger_a2c)

if if_using_ppo:
  # set up logger
  tmp_path = RESULTS_DIR + '/ppo'
  new_logger_ppo = configure(tmp_path, ["stdout", "csv", "tensorboard"])
  # Set new logger
  model_ppo.set_logger(new_logger_ppo)


trained_a2c = agent.train_model(model=model_a2c,
                             tb_log_name='a2c',
                             total_timesteps=500) if if_using_a2c else None

trained_ppo = agent.train_model(model=model_ppo,
                             tb_log_name='ppo',
                             total_timesteps=500) if if_using_ppo else None

{'n_steps': 5, 'ent_coef': 0.01, 'learning_rate': 0.0007}
Using cpu device
{'n_steps': 2048, 'ent_coef': 0.01, 'learning_rate': 0.00025, 'batch_size': 64}
Using cpu device
Logging to results/ppo
------------------------------------
| time/              |             |
|    fps             | 342         |
|    iterations      | 1           |
|    time_elapsed    | 5           |
|    total_timesteps | 2048        |
| train/             |             |
|    reward          | -0.21997054 |
------------------------------------


In [13]:
trained_a2c.save(TRAINED_MODEL_DIR + "/agent_a2c") if if_using_a2c else None
#trained_ppo.save(TRAINED_MODEL_DIR + "/agent_ppo") if if_using_ppo else None

trained_ppo.save("results/ppo/agent_ppo") if if_using_ppo else None

# Consider Joining the Newsletter
Stay Notified of New Videos and Code

https://witty-motivator-1414.ck.page/acb393f729

$$

Estás entrenando un agente de Aprendizaje por Refuerzo (RL) que aprenda a invertir en el mercado financiero automáticamente, tomando decisiones de compra, venta o mantenimiento de acciones con base en información histórica del mercado.

\section*{Resumen del Proceso en FinRL}

\textbf{Objetivo general:} Entrenar un agente de Aprendizaje por Refuerzo (RL) que aprenda a invertir en el mercado financiero tomando decisiones de compra, venta o mantenimiento de acciones con base en información histórica y señales técnicas.

\vspace{1em}
\textbf{1. Preparación de datos}

\begin{itemize}
    \item Se utiliza un \texttt{DataFrame} llamado \texttt{train} con datos históricos de precios de acciones.
    \item Incluye columnas como \texttt{date}, \texttt{tic}, \texttt{close}, indicadores técnicos como \texttt{macd}, \texttt{rsi\_30}, etc.
    \item Se aplica procesamiento con \texttt{FeatureEngineer} de FinRL.
\end{itemize}

\vspace{1em}
\textbf{2. Cálculo del espacio de estado (\texttt{state\_space})}

\begin{equation*}
    \texttt{state\_space} = 1 + 2 \times \texttt{stock\_dimension} + |\texttt{INDICATORS}| \times \texttt{stock\_dimension}
\end{equation*}

\begin{itemize}
    \item Incluye: efectivo disponible, número de acciones poseídas, precios actuales y valores de indicadores técnicos por acción.
\end{itemize}

\vspace{1em}
\textbf{3. Definición del entorno de trading}

\begin{itemize}
    \item Se instancia el entorno \texttt{StockTradingEnv} que simula el mercado financiero.
    \item Se configuran parámetros como capital inicial, comisiones, acciones disponibles, indicadores técnicos, etc.
\end{itemize}

\vspace{1em}
\textbf{4. Creación del agente (PPO o A2C)}

\begin{itemize}
    \item Se inicializa un agente con \texttt{agent.get\_model("ppo")} o \texttt{"a2c"}.
    \item Se configura un sistema de \texttt{logging} para guardar métricas y visualizaciones (consola, CSV, TensorBoard).
\end{itemize}

\vspace{1em}
\textbf{5. Entrenamiento del agente}

\begin{itemize}
    \item El agente se entrena con \texttt{agent.train\_model(...)} durante un número definido de pasos (ej. 50,000).
    \item En cada paso, el agente observa el estado del mercado, toma una acción, y recibe una recompensa.
\end{itemize}

\vspace{1em}
\textbf{6. Recompensas}

\begin{itemize}
    \item Por defecto, la recompensa en cada paso es el cambio en el valor total del portafolio:
    \[
        r_t = \text{ValorPortafolio}(t) - \text{ValorPortafolio}(t-1)
    \]
    \item Esta recompensa puede ser personalizada para incluir penalizaciones por riesgo, drawdown, volatilidad, etc.
    \item Se puede escalar con un parámetro como \texttt{reward\_scaling = 1e-4} para estabilizar el entrenamiento.
\end{itemize}

\vspace{1em}
\textbf{Siguientes pasos:}

\begin{itemize}
    \item Evaluar el desempeño del agente en un conjunto de prueba.
    \item Visualizar las decisiones tomadas y el crecimiento del portafolio.
    \item Comparar con benchmarks (ej. estrategia buy-and-hold).
    \item Ajustar hiperparámetros y recompensas según el rendimiento observado.
\end{itemize}

$$